In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

protein_path = ROOT / "data/processed/human_proteins_with_string.parquet"
gaf_path = ROOT / "data/raw/go/goa_human.gaf.gz"

protein_df = pd.read_parquet(protein_path)

GAF_COLUMNS = [
    "DB", "DB_Object_ID", "DB_Object_Symbol", "Qualifier",
    "GO_ID", "DB_Reference", "Evidence_Code", "With_or_From",
    "Aspect", "DB_Object_Name", "DB_Object_Synonym",
    "DB_Object_Type", "Taxon", "Date", "Assigned_By",
    "Annotation_Extension", "Gene_Product_Form_ID",
]

gaf_df = pd.read_csv(
    gaf_path,
    sep="\t",
    comment="!",
    header=None,
    names=GAF_COLUMNS,
    compression="gzip",
    dtype=str,
)

In [2]:
protein_ids = set(protein_df["uniprot_id"])

is_not_annotation = gaf_df["Qualifier"].fillna("").str.contains(
    r"(?:^|\|)NOT(?:\||$)",
    regex=True,
)

mf_annotations = gaf_df[
    (~is_not_annotation)
    & (gaf_df["Aspect"] == "F")
    & (gaf_df["DB_Object_ID"].isin(protein_ids))
][
    ["DB_Object_ID", "GO_ID"]
].drop_duplicates().rename(
    columns={
        "DB_Object_ID": "uniprot_id",
        "GO_ID": "go_id",
    }
)

term_support = (
    mf_annotations.groupby("go_id")["uniprot_id"]
    .nunique()
    .sort_values(ascending=False)
)

In [3]:
MIN_PROTEINS_PER_TERM = 100

# These labels are too broad for a meaningful first protein-function benchmark.
EXCLUDED_GO_TERMS = {
    "GO:0005515",  # protein binding
    "GO:0042802",  # identical protein binding
}

target_terms = (
    term_support[term_support >= MIN_PROTEINS_PER_TERM]
    .drop(labels=EXCLUDED_GO_TERMS, errors="ignore")
    .rename("protein_count")
    .reset_index()
)

target_terms["label_index"] = range(len(target_terms))

print("Final target GO terms:", len(target_terms))
target_terms.head(15)

Final target GO terms: 96


,go_id,protein_count,label_index
0,GO:0003723,1386,0
1,GO:0000981,1197,1
2,GO:0003677,1176,2
3,GO:0005524,1137,3
4,GO:0000978,1085,4
5,GO:0008270,841,5
6,GO:0004930,759,6
7,GO:0003700,716,7
8,GO:0042803,716,8
9,GO:0005509,693,9


In [4]:
selected_mf_labels = mf_annotations.merge(
    target_terms[["go_id", "label_index"]],
    on="go_id",
    how="inner",
)

labelled_proteins = selected_mf_labels["uniprot_id"].nunique()

labels_per_protein = (
    selected_mf_labels.groupby("uniprot_id")["go_id"]
    .nunique()
)

print("Protein-GO label pairs:", f"{len(selected_mf_labels):,}")
print("Proteins with at least one selected label:", f"{labelled_proteins:,}")
print("Proteins without a selected label:", f"{len(protein_df) - labelled_proteins:,}")

print("\nSelected labels per protein:")
print(labels_per_protein.describe())

Protein-GO label pairs: 29,730
Proteins with at least one selected label: 12,020
Proteins without a selected label: 6,821

Selected labels per protein:
count    12020.000000
mean         2.473378
std          1.942095
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         18.000000
Name: go_id, dtype: float64


In [5]:
processed_dir = ROOT / "data/processed"

labels_output_path = processed_dir / "go_mf_labels_min_support_100.parquet"
terms_output_path = processed_dir / "go_mf_target_terms_min_support_100.csv"

selected_mf_labels.to_parquet(labels_output_path, index=False)
target_terms.to_csv(terms_output_path, index=False)

print("Saved labels:", labels_output_path)
print("Saved target terms:", terms_output_path)

Saved labels: c:\Users\rajdeep\Desktop\Capstone Project_GNN\data\processed\go_mf_labels_min_support_100.parquet
Saved target terms: c:\Users\rajdeep\Desktop\Capstone Project_GNN\data\processed\go_mf_target_terms_min_support_100.csv
